# rotary position embedding

In [1]:
!pip install torch;

Looking in indexes: http://mirrors.cloud.aliyuncs.com/pypi/simple/


## 1. math from paper
basic equation
$$f(q,m)=
\begin{pmatrix}
 \cos m\theta & -\sin m\theta  \\
 \sin m\theta & \cos m\theta
\end{pmatrix}
\begin{pmatrix}
 q_{0}  \\
 q_{1} 
\end{pmatrix}
$$

to expand to full matrix

$$
=\begin{pmatrix}
 \cos m\theta_{0} & -\sin m\theta_{0} & 0 & 0 & \cdots & 0 & 0 \\
 \sin m\theta_{0} & \cos m\theta_{0} & 0 & 0 & \cdots & 0 & 0 \\
 0 & 0 & \cos m\theta_{1} & -\sin m\theta_{1} & \cdots & 0 & 0 \\
 0 & 0 & \sin m\theta_{1} & \cos m\theta_{1} & \cdots  & 0 & 0 \\
 \vdots &  \vdots &  \vdots &  \vdots & \ddots & \vdots & \vdots \\
 0 & 0 & 0 & 0 & \cdots & \cos m\theta_{d/2} & -\sin m\theta_{d/2} \\
 0 & 0 & 0 & 0 & \cdots & \sin m\theta_{d/2} & \cos m\theta_{d/2}\\
\end{pmatrix}
\begin{pmatrix}
 q_{0} \\
 q_{1} \\
 q_{2} \\
 q_{3} \\
 \vdots \\
 q_{d/2-2} \\
 q_{d/2-1} \\
\end{pmatrix}
$$

with rewrite to be computational efficient

$$
\begin{pmatrix}
 q_{0} \\
 q_{1} \\
 q_{2} \\
 q_{3} \\
 \vdots \\
 q_{d-2} \\
 q_{d-1} \\
\end{pmatrix}
\bigotimes
\begin{pmatrix}
 cos m\theta_{0} \\
 cos m\theta_{0} \\
 cos m\theta_{1} \\
 cos m\theta_{1} \\
 \vdots \\
 cos m\theta_{d/2-1} \\
 cos m\theta_{d/2-1} \\
\end{pmatrix}
+
\begin{pmatrix}
 -q_{1} \\
 q_{0} \\
 -q_{3} \\
 q_{2} \\
 \vdots \\
 -q_{d-1} \\
 q_{d-2} \\
\end{pmatrix}
\bigotimes
\begin{pmatrix}
 sin m\theta_{0} \\
 sin m\theta_{0} \\
 sin m\theta_{1} \\
 sin m\theta_{1} \\
 \vdots \\
 sin m\theta_{d/2-1} \\
 sin m\theta_{d/2-1} \\
\end{pmatrix}
$$

In [2]:
import torch
import numpy as np

# config
dim = 4096 # also embed_size
max_seq_len = 2048 # 最长输入长度
bsz = 16 # batch_size
seq_len = 6 # 当前输入长度
num_heads = 32 #
head_dim = dim // num_heads # 128

## 2. implementation at roformer (rope original paper)

cal sinusoidal position

https://github.com/JunnYu/RoFormer_pytorch/blob/roformer_v2/src/roformer/modeling_roformer.py#L156

apply rotary

https://github.com/JunnYu/RoFormer_pytorch/blob/roformer_v2/src/roformer/modeling_roformer.py#L441

the code follows math equation by rotating in pairs

In [3]:
import torch

def roformer_apply_rotary(x):
    
    # RoFormerSinusoidalPositionalEmbedding
    position_enc = np.array(
        [
            [pos / np.power(10000, 2 * (j // 2) / head_dim) for j in range(head_dim)]
            for pos in range(seq_len)
        ]
    )
    sin = torch.FloatTensor(np.sin(position_enc[:, 0::2]))
    cos = torch.FloatTensor(np.cos(position_enc[:, 1::2]))

#     sentinel = dim // 2 if dim % 2 == 0 else (dim // 2) + 1
#     out[:, 0:sentinel] = torch.FloatTensor(np.sin(position_enc[:, 0::2]))
#     out[:, sentinel:] = torch.FloatTensor(np.cos(position_enc[:, 1::2]))
    out = torch.cat((sin,cos),dim=-1) # [seq_len,head_dim]
    
    # [sequence_length, embed_size_per_head] -> sin & cos [batch_size, num_heads, sequence_length, embed_size_per_head // 2] 
    sin, cos = out[None,None,:,:].chunk(2, dim=-1)
    
#     same as
#     out = torch.cat((sin,cos),dim=-1).repeat(bsz,num_heads,1,1) # [seq_len,head_dim]
#     sin, cos = out.chunk(2, dim=-1)
    
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).flatten(-2, -1)

## 3. implementation at GPTNeo
https://github.com/EleutherAI/gpt-neo/blob/master/models/layers.py#L355

this is **different** from paper cause it uses rotate_half instead of rotate_every_two

confirmed by https://github.com/EleutherAI/gpt-neox/pull/241 and discussed in https://github.com/EleutherAI/gpt-neox/issues/834 that it is efficient without performance loss

## 4. implementation at facebook llama

llama paper indicates that it is inspired from GPTNeo

but the rope use complex number trick that actually follow the original paper, NOT GPTNeo rotate_half

https://github.com/facebookresearch/llama/blob/main/llama/model.py#L63

In [4]:
def llama_apply_rotary_emb(x):
    # L123 llama use x shape (bsz, seqlen, self.n_local_heads, self.head_dim)
    # so here transpose back
    x = x.transpose(1, 2) # (bsz, seqlen, num_heads, head_dim)
    
    # precompute_freqs_cis
    theta = 10000
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))
    t = torch.arange(max_seq_len*2)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    
    # Transformer.forward after tok_embeddings
    freqs_cis = freqs_cis[0:seq_len] # (seq_len, head_dim/2)
    
    
    # apply_rotary_emb
    x = x.float().reshape(*x.shape[:-1], -1, 2) # last dimension [x0,x1,x2,..] to [[x0,x1],[x2,]..] pairs, (bsz, seqlen, num_heads, head_dim/2, 2)
    x = torch.view_as_complex(x) # (bsz, seqlen, num_heads, head_dim/2)
    
    ndim = x.ndim
    assert 0 <= 1 < ndim
    assert freqs_cis.shape == (x.shape[1], x.shape[-1])
    shape = [d if i == 1 or i == x.ndim - 1 else 1 for i, d in enumerate(x.shape)]
    freqs_cis =freqs_cis.view(*shape) # (1, seq_len, 1, head_dim/2)
   
    o = torch.view_as_real(x * freqs_cis) # (bsz, seq_len, num_heads, head_dim/2, 2)
    o = o.flatten(3) # (bsz, seq_len, num_heads, head_dim)
    return o.transpose(1, 2) # (bsz, num_heads, seq_len, head_dim)


## 5. implementation at transformers Llama

https://github.com/fpgaminer/transformers/blob/main/src/transformers/models/llama/modeling_llama.py

In [7]:
def transformers_apply_rotary_pos_emb(x):
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    # LlamaRotaryEmbedding
    inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float()/ head_dim)) # (max_seq_len)
    t = torch.arange(max_seq_len, dtype=inv_freq.dtype)
    freqs = torch.einsum("i,j->ij", t, inv_freq) # (max_seq_len,max_seq_len)
    emb = torch.cat((freqs, freqs), dim=-1)
    cos_cached = emb.cos()[None, None, :, :]
    sin_cached = emb.sin()[None, None, :, :]

    # apply_rotary_pos_emb
    cos = cos_cached[:,:,:seq_len,...]
    sin = sin_cached[:,:,:seq_len,...]
    cos = cos.squeeze(1).squeeze(0)  
    sin = sin.squeeze(1).squeeze(0)  
    position_ids = torch.arange(0, seq_len).repeat(bsz) #(bsz, seq_len)
    position_ids = position_ids.unsqueeze(0).view(-1, seq_len) 
    cos = cos[position_ids].unsqueeze(1)  # (bsz, 1, seq_len, dim)
    sin = sin[position_ids].unsqueeze(1)  # (bsz, 1, seq_len, dim)
    return (x * cos) + (rotate_half(x) * sin)


## test RoPE result


roformer= facebook llama!= transformers llama / GPTNeo

but using transformers LlamaForCausalLM for inference actually works the same as facebook llama

according to https://discuss.huggingface.co/t/why-llama-weight-in-huggingface-need-to-do-permute-on-wq-wk/37643

the magic is this permute during weight conversion

In [8]:
x = torch.randn((bsz, seq_len, dim))
q = torch.nn.Linear(4096,4096,bias=False)
query_states = q(x).view(bsz, seq_len, num_heads, head_dim).transpose(1, 2) # (bsz, num_heads, seq_len, head_dim)

e1 = roformer_apply_rotary(torch.clone(query_states))
e2 = llama_apply_rotary_emb(torch.clone(query_states))
e3 = transformers_apply_rotary_pos_emb(torch.clone(query_states))

print(torch.allclose(e1,e2,atol=1e-6))
print(torch.allclose(e2,e3,atol=1e-4))
print(torch.allclose(e1,e3,atol=1e-4))

True
False
False
